In [ ]:
# Cell 1: Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

In [ ]:
# Cell 2: Configuration & Transforms
# FER2013 is 48x48 Grayscale, but ResNet wants 224x224 RGB.
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),        # Resize to standard model input
        transforms.Grayscale(num_output_channels=3), # Fake RGB by stacking 3 channels
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# UPDATE THIS PATH to where your data is
data_dir = './data' 
image_datasets = {x: datasets.ImageFolder(f'{data_dir}/{x}', data_transforms[x]) for x in ['train', 'test']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=32, shuffle=True) for x in ['train', 'test']}
class_names = image_datasets['train'].classes
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
model = models.densenet121(pretrained=True)

# Modify the Classifier layer for 7 emotions
# DenseNet's classifier is called 'classifier', not 'fc'
num_ftrs = model.classifier.in_features
model.classifier = nn.Linear(num_ftrs, 7)
model = model.to(device)

In [ ]:
# Cell 4: Training Setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)


In [ ]:
# Cell 5: Training Loop
def train_model(model, criterion, optimizer, epochs=5):
    for epoch in range(epochs):
        print(f'Epoch {epoch+1}/{epochs}')
        
        for phase in ['train', 'test']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')


In [ ]:
# Cell 6: Run Training
train_model(model, criterion, optimizer, epochs=10)

In [ ]:
# Cell 7: Save Model
torch.save(model.state_dict(), 'resnet18_fer2013.pth')
print("Model saved successfully!")